# SADAR Merchant Classification — Final Production One Notebook Generator

Notebook ini adalah versi final yang mengikuti konsep production-ready:

- **Deep Learning TensorFlow Functional API** menjadi model utama untuk prediksi `category_detail`.
- **Dictionary CSV** hanya menjadi source of truth manusia saat training.
- Notebook akan meng-compile dictionary CSV menjadi **`artifacts/dictionary_rules.json`** untuk production.
- File `.keras` **tidak membaca CSV/JSON secara langsung**. Yang membaca `.keras` dan `dictionary_rules.json` adalah **`inference.py`**.
- `category_primary` tidak diprediksi AI, tetapi ditentukan menggunakan **`category_primary_map.json`** agar konsisten dengan aturan bisnis.

Output akhir notebook adalah folder deployment:

```text
/content/sadar_merchant_classifier_production/
├── main.py
├── inference.py
├── requirements.txt
├── README.md
├── .python-version
├── .gitignore
├── model/
│   ├── sadar_merchant_detail_classifier.keras
│   └── sadar_merchant_detail_classifier_savedmodel/
├── artifacts/
│   ├── dictionary_rules.json
│   ├── category_primary_map.json
│   ├── label_classes.json
│   └── metadata.json
├── dictionary/
│   └── merchant_dictionary_validator_enhanced.csv
└── data/
    └── sample_transactions.csv
```

Folder itulah yang nanti masuk GitHub / Render / Railway.


## 1. Install dependency tambahan Colab-safe

In [1]:
# Colab biasanya sudah punya TensorFlow, pandas, numpy, scikit-learn.
# Jangan reinstall TensorFlow/Pandas/Numpy di Colab karena bisa memicu binary incompatibility.
!pip -q install rapidfuzz tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 34.2 MB/s eta 0:00:00


## 2. Path file dari Google Drive / fallback /content

In [2]:
from pathlib import Path

USE_GOOGLE_DRIVE = True

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        print('Drive mount dilewati:', e)

# Ubah folder ini sesuai lokasi file kamu di Google Drive.
DRIVE_BASE_DIR = Path('/content/drive/MyDrive/SADAR_Finance_Model/Merch')
LOCAL_BASE_DIR = Path('/content')

DATA_CANDIDATES = [
    DRIVE_BASE_DIR / 'data_budget_modelling.csv',
    DRIVE_BASE_DIR / 'data_modelling.csv',
    LOCAL_BASE_DIR / 'data_budget_modelling.csv',
    LOCAL_BASE_DIR / 'data_modelling.csv',
]

DICTIONARY_CANDIDATES = [
    # Paling direkomendasikan: dictionary generic v7 yang sudah mencakup merchant umum
    DRIVE_BASE_DIR / 'merchant_dictionary_validator_generic_v7.csv',
    DRIVE_BASE_DIR / 'merchant_dictionary_validator_enhanced.csv',
    DRIVE_BASE_DIR / 'merchant_dictionary_validator.csv',
    DRIVE_BASE_DIR / 'sadar_indonesia_rules_hybrid_v6_dataset_expanded.csv',
    LOCAL_BASE_DIR / 'merchant_dictionary_validator_generic_v7.csv',
    LOCAL_BASE_DIR / 'merchant_dictionary_validator_enhanced.csv',
    LOCAL_BASE_DIR / 'merchant_dictionary_validator.csv',
    LOCAL_BASE_DIR / 'sadar_indonesia_rules_hybrid_v6_dataset_expanded.csv',
]

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

DATA_PATH = first_existing(DATA_CANDIDATES)
DICTIONARY_CSV_PATH = first_existing(DICTIONARY_CANDIDATES)

if DATA_PATH is None:
    raise FileNotFoundError('Dataset tidak ditemukan. Simpan CSV di Drive atau upload ke /content.')
if DICTIONARY_CSV_PATH is None:
    raise FileNotFoundError('Dictionary tidak ditemukan. Simpan CSV di Drive atau upload ke /content.')

print('DATA_PATH:', DATA_PATH)
print('DICTIONARY_CSV_PATH:', DICTIONARY_CSV_PATH)

OUTPUT_DIR = Path('/content/sadar_merchant_classifier_production')
MODEL_DIR = OUTPUT_DIR / 'model'
ARTIFACTS_DIR = OUTPUT_DIR / 'artifacts'
DICTIONARY_DIR = OUTPUT_DIR / 'dictionary'
DATA_SAMPLE_DIR = OUTPUT_DIR / 'data'

for d in [OUTPUT_DIR, MODEL_DIR, ARTIFACTS_DIR, DICTIONARY_DIR, DATA_SAMPLE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('OUTPUT_DIR:', OUTPUT_DIR)

Mounted at /content/drive
DATA_PATH: /content/drive/MyDrive/SADAR_Finance_Model/Merch/data_budget_modelling.csv
DICTIONARY_CSV_PATH: /content/drive/MyDrive/SADAR_Finance_Model/Merch/merchant_dictionary_validator.csv
OUTPUT_DIR: /content/sadar_merchant_classifier_production


### Catatan path penting

- `DATA_PATH` dan `DICTIONARY_CSV_PATH` boleh berasal dari Google Drive karena itu hanya untuk **training di Colab**.
- Setelah training selesai, notebook menghasilkan folder `/content/sadar_merchant_classifier_production`.
- Di production, API tidak membaca Google Drive dan tidak memakai `files.upload()`.
- `inference.py` membaca file dengan relative path dari folder deployment:

```text
model/sadar_merchant_detail_classifier.keras
artifacts/dictionary_rules.json
artifacts/category_primary_map.json
artifacts/label_classes.json
artifacts/metadata.json
```


## 3. Import dan konfigurasi global

In [3]:
import os
import re
import json
import math
import shutil
import random
import unicodedata
from datetime import datetime
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

TRUSTED_RULE_THRESHOLD = 0.90
MODEL_REVIEW_THRESHOLD = 0.60

CATEGORY_PRIMARY_MAP = {
    'food': 'Needs',
    'groceries': 'Needs',
    'health': 'Needs',
    'transport': 'Needs',
    'utilities': 'Needs',
    'education': 'Wants',
    'entertainment': 'Wants',
    'self_care': 'Wants',
    'shopping': 'Wants',
    'travel': 'Wants',
    'investment': 'Investment',
}

CATEGORICAL_FEATURES = [
    'payment_method',
    'payment_media',
    'source',
    'day_of_week',
    'time_of_day',
    'spending_level',
    'is_weekend',
]

NUMERIC_FEATURES = [
    'amount',
    'log_amount',
    'month',
    'day',
    'week',
    'rolling_7d_spending',
    'rolling_30d_spending',
    'transaction_count_to_date',
]

MODEL_KERAS_FILE = 'sadar_merchant_detail_classifier.keras'
MODEL_SAVEDMODEL_DIR = 'sadar_merchant_detail_classifier_savedmodel'

print('TensorFlow:', tf.__version__)

TensorFlow: 2.20.0


## 4. Utility normalisasi teks

In [4]:
def normalize_text(text):
    if pd.isna(text):
        return 'unknown'
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else 'unknown'

def split_pipe(value):
    if pd.isna(value):
        return []
    parts = [normalize_text(x) for x in str(value).split('|')]
    return [x for x in parts if x and x != 'unknown']

def normalize_enum(value, default='unknown'):
    """Normalizer untuk nilai enum seperti starts_with, review_required, never_override.
    Berbeda dari normalize_text karena underscore tetap dipertahankan.
    """
    if pd.isna(value):
        return default
    text = str(value).strip().lower()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9_\s\-]', ' ', text)
    text = re.sub(r'[\s\-]+', '_', text)
    text = re.sub(r'_+', '_', text).strip('_')
    return text if text else default

def safe_float(value, default=None):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default

def infer_spending_level(amount):
    amount = float(amount or 0)
    if amount < 50_000:
        return 'low'
    if amount < 300_000:
        return 'medium'
    return 'high'

def infer_time_of_day(hour):
    try:
        hour = int(hour)
    except Exception:
        hour = 0
    if 5 <= hour < 11:
        return 'pagi'
    if 11 <= hour < 15:
        return 'siang'
    if 15 <= hour < 18:
        return 'sore'
    return 'malam'

## 5. Load dan compile dictionary menjadi production rules

In [5]:
dictionary_df = pd.read_csv(DICTIONARY_CSV_PATH)
dictionary_df.columns = dictionary_df.columns.str.strip()

required_dict_cols = ['alias_keywords', 'category_detail']
for col in required_dict_cols:
    if col not in dictionary_df.columns:
        raise ValueError(f'Kolom dictionary wajib tidak ada: {col}')

# Lengkapi kolom opsional agar robust untuk berbagai versi dictionary.
def ensure_col(df, col, default):
    if col not in df.columns:
        df[col] = default

optional_defaults = {
    'rule_id': '',
    'active': True,
    'priority': 50,
    'match_type': 'contains',
    'canonical_merchant': '',
    'negative_keywords': '',
    'category_primary': '',
    'merchant_type': '',
    'rule_strength': 'medium',
    'ambiguity_level': 'medium',
    'validator_action': 'override_if_model_low_confidence',
    'min_model_confidence_accept': 0.75,
    'max_model_confidence_override': 0.85,
    'amount_min': np.nan,
    'amount_max': np.nan,
    'amount_action': 'warning',
    'payment_method_hint': '',
    'payment_media_hint': '',
    'source_hint': '',
    'notes': '',
    'version': 'v1',
}

for col, default in optional_defaults.items():
    ensure_col(dictionary_df, col, default)

def normalize_bool(x):
    if isinstance(x, bool):
        return x
    if pd.isna(x):
        return True
    return str(x).strip().lower() in {'true', '1', 'yes', 'y', 'active'}

def rule_confidence_from_row(row):
    strength = normalize_enum(row.get('rule_strength', 'medium'), 'medium')
    ambiguity = normalize_enum(row.get('ambiguity_level', 'medium'), 'medium')
    priority = safe_float(row.get('priority', 50), 50)

    base = {
        'very_strong': 0.96,
        'strong': 0.92,
        'medium': 0.78,
        'weak': 0.60,
    }.get(strength, 0.75)

    if ambiguity == 'low':
        base += 0.04
    elif ambiguity == 'high':
        base -= 0.12
    elif ambiguity == 'medium':
        base -= 0.03

    base += min(max((priority - 50) / 1000, -0.05), 0.05)
    return float(np.clip(base, 0.05, 0.99))

def compile_dictionary_rules(dictionary_df):
    rules = []
    for _, row in dictionary_df.iterrows():
        active = normalize_bool(row.get('active', True))
        aliases = split_pipe(row.get('alias_keywords', ''))
        negatives = split_pipe(row.get('negative_keywords', ''))
        detail = normalize_text(row.get('category_detail', ''))

        if not active or not aliases or detail in {'', 'unknown', 'nan'}:
            continue

        primary = str(row.get('category_primary', '')).strip()
        if not primary or primary.lower() == 'nan':
            primary = CATEGORY_PRIMARY_MAP.get(detail, 'unknown')

        rule = {
            'rule_id': str(row.get('rule_id', '')).strip() or f'RULE_{len(rules)+1:05d}',
            'active': active,
            'priority': int(safe_float(row.get('priority', 50), 50)),
            'match_type': normalize_enum(row.get('match_type', 'contains'), 'contains'),
            'canonical_merchant': str(row.get('canonical_merchant', '')).strip(),
            'alias_keywords': aliases,
            'negative_keywords': negatives,
            'category_detail': detail,
            'category_primary': primary,
            'merchant_type': str(row.get('merchant_type', '')).strip(),
            'rule_strength': normalize_enum(row.get('rule_strength', 'medium'), 'medium'),
            'ambiguity_level': normalize_enum(row.get('ambiguity_level', 'medium'), 'medium'),
            'validator_action': normalize_enum(row.get('validator_action', 'override_if_model_low_confidence'), 'override_if_model_low_confidence'),
            'rule_confidence': rule_confidence_from_row(row),
            'min_model_confidence_accept': safe_float(row.get('min_model_confidence_accept', 0.75), 0.75),
            'max_model_confidence_override': safe_float(row.get('max_model_confidence_override', 0.85), 0.85),
            'amount_min': safe_float(row.get('amount_min', np.nan), None),
            'amount_max': safe_float(row.get('amount_max', np.nan), None),
            'amount_action': normalize_enum(row.get('amount_action', 'warning'), 'warning'),
            'notes': str(row.get('notes', '')).strip(),
            'version': str(row.get('version', 'v1')).strip(),
        }

        rule['is_trusted_guardrail'] = (
            rule['rule_confidence'] >= TRUSTED_RULE_THRESHOLD
            and rule['rule_strength'] in {'strong', 'very_strong'}
            and rule['ambiguity_level'] == 'low'
            and rule['validator_action'] not in {'never_override', 'review_required'}
        )
        rules.append(rule)

    rules = sorted(rules, key=lambda r: (r['priority'], r['rule_confidence']), reverse=True)
    return rules

DICTIONARY_RULES = compile_dictionary_rules(dictionary_df)
print('Raw dictionary rows:', len(dictionary_df))
print('Compiled active rules:', len(DICTIONARY_RULES))
print('Trusted guardrail rules:', sum(r['is_trusted_guardrail'] for r in DICTIONARY_RULES))

# Preview rules
pd.DataFrame(DICTIONARY_RULES).head(10)[['rule_id','priority','category_detail','rule_confidence','is_trusted_guardrail','alias_keywords']]

Raw dictionary rows: 959
Compiled active rules: 959
Trusted guardrail rules: 828


,rule_id,priority,category_detail,rule_confidence,is_trusted_guardrail,alias_keywords
0,GENERIC_EDUCATION_SPP_UKT_SEKOLAH_KULIAH,99,education,0.99,True,"[spp, ukt, uang kuliah, biaya kuliah, biaya se..."
1,GENERIC_EDUCATION_TOKO_BUKU_ATK,99,education,0.99,True,"[toko buku, bookstore, toko atk, toko alat tul..."
2,GENERIC_ENTERTAINMENT_STREAMING_SUBSCRIPTION,99,entertainment,0.99,True,"[netflix, spotify, youtube premium, disney plu..."
3,FOOD_KOPI_CAFE_BEVERAGE_GENERIC_ENHANCED,99,food,0.99,True,"[kopi, coffee, coffe, cofee, cafe, cafe, kafe,..."
4,FOOD_RESTO_RESTAURANT_GENERIC_ENHANCED,99,food,0.99,True,"[restoran, restaurant, resto, rumah makan, rm,..."
5,GENERIC_FOOD_KOPERASI_KANTIN,99,food,0.99,True,"[kantin koperasi, koperasi kantin, koperasi ma..."
6,GENERIC_FOOD_KOPI_CAFE_EXPANDED,99,food,0.99,True,"[kopi, coffee, cafe, kafe, kedai kopi, warung ..."
7,GENERIC_FOOD_MENU_INDONESIA_EXPANDED,99,food,0.99,True,"[nasi padang, nasi goreng, nasi uduk, nasi kun..."
8,GENERIC_FOOD_RESTO_WARUNG_EXPANDED,99,food,0.99,True,"[warung makan, rumah makan, restoran, restaura..."
9,GENERIC_FOOD_TOKO_ROTI_KUE,99,food,0.99,True,"[toko roti, toko kue, kedai roti, kedai kue, w..."


## 6. Rule matcher untuk inference dan guardrail

In [6]:
def amount_allowed(rule, amount):
    amount = float(amount or 0)
    amin = rule.get('amount_min')
    amax = rule.get('amount_max')
    if amin is not None and amount < amin:
        return False
    if amax is not None and amount > amax:
        return False
    return True

def keyword_matches(merchant_clean, keyword, match_type):
    if not keyword:
        return False
    if match_type == 'exact':
        return merchant_clean == keyword
    if match_type == 'starts_with':
        return merchant_clean.startswith(keyword)
    # Default contains dengan word boundary ringan untuk keyword pendek.
    if len(keyword) <= 3:
        return re.search(rf'(?<![a-z0-9]){re.escape(keyword)}(?![a-z0-9])', merchant_clean) is not None
    return keyword in merchant_clean

def apply_dictionary_rules(merchant, amount=0):
    merchant_clean = normalize_text(merchant)
    best = None

    for rule in DICTIONARY_RULES:
        if not rule.get('active', True):
            continue

        # Negative keyword membuat rule tidak aman.
        if any(keyword_matches(merchant_clean, neg, 'contains') for neg in rule.get('negative_keywords', [])):
            continue

        if not amount_allowed(rule, amount):
            continue

        match_type = rule.get('match_type', 'contains')
        matched_keyword = None
        for kw in rule.get('alias_keywords', []):
            if keyword_matches(merchant_clean, kw, match_type):
                matched_keyword = kw
                break

        if matched_keyword:
            best = {
                'rule_category': rule['category_detail'],
                'rule_primary': rule.get('category_primary', CATEGORY_PRIMARY_MAP.get(rule['category_detail'], 'unknown')),
                'rule_confidence': float(rule['rule_confidence']),
                'matched_rule': rule['rule_id'],
                'matched_keyword': matched_keyword,
                'is_trusted_guardrail': bool(rule['is_trusted_guardrail']),
                'rule_ambiguity_level': rule['ambiguity_level'],
                'rule_strength': rule['rule_strength'],
            }
            break

    if best is None:
        best = {
            'rule_category': None,
            'rule_primary': None,
            'rule_confidence': 0.0,
            'matched_rule': None,
            'matched_keyword': None,
            'is_trusted_guardrail': False,
            'rule_ambiguity_level': None,
            'rule_strength': None,
        }
    return best

# Cek cepat beberapa merchant baru, bukan full dataset.
for m in ['Kopi Senja', 'Cafe Nusantara', 'Warung Makan Bu Ani', 'Toko Berkah', 'PT Maju Sejahtera']:
    print(m, '=>', apply_dictionary_rules(m, 35000))

Kopi Senja => {'rule_category': 'food', 'rule_primary': 'Needs', 'rule_confidence': 0.99, 'matched_rule': 'FOOD_KOPI_CAFE_BEVERAGE_GENERIC_ENHANCED', 'matched_keyword': 'kopi', 'is_trusted_guardrail': True, 'rule_ambiguity_level': 'low', 'rule_strength': 'strong'}
Cafe Nusantara => {'rule_category': 'food', 'rule_primary': 'Needs', 'rule_confidence': 0.99, 'matched_rule': 'FOOD_KOPI_CAFE_BEVERAGE_GENERIC_ENHANCED', 'matched_keyword': 'cafe', 'is_trusted_guardrail': True, 'rule_ambiguity_level': 'low', 'rule_strength': 'strong'}
Warung Makan Bu Ani => {'rule_category': 'food', 'rule_primary': 'Needs', 'rule_confidence': 0.99, 'matched_rule': 'GENERIC_FOOD_RESTO_WARUNG_EXPANDED', 'matched_keyword': 'warung makan', 'is_trusted_guardrail': True, 'rule_ambiguity_level': 'low', 'rule_strength': 'strong'}
Toko Berkah => {'rule_category': 'shopping', 'rule_primary': 'Wants', 'rule_confidence': 0.66, 'matched_rule': 'GENERIC_AMBIGUOUS_TOKO_WARUNG_BASE_REVIEW', 'matched_keyword': 'toko', 'is_tru

## 7. Load dataset dan feature engineering tanpa rules full-dataset

In [7]:
df_raw = pd.read_csv(DATA_PATH)
df_raw.columns = df_raw.columns.str.strip()

for col in ['merchant', 'amount', 'category_detail']:
    if col not in df_raw.columns:
        raise ValueError(f'Kolom wajib tidak ada: {col}')

if 'transaction_id' in df_raw.columns:
    before = len(df_raw)
    df_raw = df_raw.drop_duplicates(subset=['transaction_id']).reset_index(drop=True)
    print(f'Drop duplicate transaction_id: {before} -> {len(df_raw)}')

df_raw['category_detail'] = df_raw['category_detail'].astype(str).str.strip().str.lower()
df_raw = df_raw[~df_raw['category_detail'].isin(['nan', 'unknown', 'unknown_need_review'])].copy()

VALID_CATEGORIES = sorted(df_raw['category_detail'].dropna().unique().tolist())
print('Valid categories:', VALID_CATEGORIES)
print('Dataset:', df_raw.shape)

def prepare_features(df):
    df = df.copy()
    df['merchant'] = df['merchant'].fillna('unknown').astype(str)
    df['merchant_clean'] = df['merchant'].apply(normalize_text)
    df['amount'] = pd.to_numeric(df['amount'], errors='coerce').fillna(0.0)
    df['log_amount'] = np.log1p(df['amount'].clip(lower=0))

    if 'date' in df.columns:
        dt = pd.to_datetime(df['date'], errors='coerce')
        df['month'] = dt.dt.month.fillna(0).astype(int)
        df['day'] = dt.dt.day.fillna(0).astype(int)
        df['week'] = dt.dt.isocalendar().week.astype('float').fillna(0).astype(int)
        df['day_of_week'] = dt.dt.day_name().fillna('unknown').str.lower()
        df['is_weekend'] = (dt.dt.dayofweek >= 5).fillna(False).astype(str).str.lower()
        df['time_of_day'] = dt.dt.hour.fillna(0).apply(infer_time_of_day)
    else:
        df['month'] = 0
        df['day'] = 0
        df['week'] = 0
        df['day_of_week'] = 'unknown'
        df['is_weekend'] = 'false'
        df['time_of_day'] = 'unknown'

    for col in ['payment_method', 'payment_media', 'source']:
        if col not in df.columns:
            df[col] = 'unknown'
        df[col] = df[col].fillna('unknown').astype(str).str.lower().apply(normalize_text)

    df['spending_level'] = df['amount'].apply(infer_spending_level)

    for col in ['rolling_7d_spending', 'rolling_30d_spending', 'transaction_count_to_date']:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

    df['sample_origin'] = df.get('sample_origin', 'real')
    df['base_sample_weight'] = df.get('base_sample_weight', 1.0)
    return df

df_features = prepare_features(df_raw)

display(df_features[['merchant','merchant_clean','amount','category_detail','payment_method','payment_media','source','spending_level']].head(10))
print(df_features['category_detail'].value_counts())

Drop duplicate transaction_id: 36783 -> 9254
Valid categories: ['education', 'entertainment', 'food', 'groceries', 'health', 'investment', 'self_care', 'shopping', 'transport', 'travel', 'utilities']
Dataset: (9254, 41)


,merchant,merchant_clean,amount,category_detail,payment_method,payment_media,source,spending_level
0,Tokocrypto,tokocrypto,1835000.0,investment,credit card,bni,scholarship,high
1,Apotek K24,apotek k24,18000.0,health,debit,cimb niaga,scholarship,low
2,Compass,compass,587000.0,shopping,credit card,cimb niaga,scholarship,high
3,Indodax,indodax,456500.0,investment,transfer,linkaja,scholarship,high
4,Pintu Crypto,pintu crypto,1689000.0,investment,qris,bca,scholarship,high
5,PLN Mobile,pln mobile,337500.0,utilities,credit card,bni,scholarship,high
6,Zenius,zenius,209000.0,education,transfer,bca,scholarship,medium
7,Dicoding,dicoding,155500.0,education,transfer,ovo,scholarship,medium
8,Udemy,udemy,239500.0,education,cash,cash,scholarship,medium
9,Klinik Pratama,klinik pratama,357000.0,health,transfer,linkaja,scholarship,high


category_detail
education        898
shopping         897
investment       870
groceries        854
transport        848
food             839
travel           836
utilities        828
health           809
self_care        794
entertainment    781
Name: count, dtype: int64


## 8. Cold merchant split: evaluasi merchant baru yang lebih jujur

In [8]:
def cold_merchant_split(df, test_size=0.2, seed=42):
    df = df.copy().reset_index(drop=True)
    merchant_label = (
        df.groupby('merchant_clean')['category_detail']
          .agg(lambda s: s.value_counts().index[0])
          .reset_index()
    )

    labels = merchant_label['category_detail']
    can_stratify = labels.value_counts().min() >= 2

    train_merchants, test_merchants = train_test_split(
        merchant_label['merchant_clean'],
        test_size=test_size,
        random_state=seed,
        stratify=labels if can_stratify else None,
    )

    train_df = df[df['merchant_clean'].isin(set(train_merchants))].reset_index(drop=True)
    test_df = df[df['merchant_clean'].isin(set(test_merchants))].reset_index(drop=True)
    return train_df, test_df

train_val_df, cold_test_df = cold_merchant_split(df_features, test_size=0.20, seed=SEED)
train_df, val_df = cold_merchant_split(train_val_df, test_size=0.15, seed=SEED + 1)

print('Train:', train_df.shape, 'unique merchants:', train_df['merchant_clean'].nunique())
print('Val  :', val_df.shape, 'unique merchants:', val_df['merchant_clean'].nunique())
print('Cold :', cold_test_df.shape, 'unique merchants:', cold_test_df['merchant_clean'].nunique())

print('\nCold merchant overlap train-test:', len(set(train_df['merchant_clean']) & set(cold_test_df['merchant_clean'])))

Train: (6202, 51) unique merchants: 126
Val  : (1139, 51) unique merchants: 23
Cold : (1913, 51) unique merchants: 38

Cold merchant overlap train-test: 0


## 9. Synthetic augmentation dari dictionary + typo/noise augmentation

In [9]:
def sample_amount_for_rule(rule):
    amin = rule.get('amount_min')
    amax = rule.get('amount_max')
    if amin is None and amax is None:
        return 50_000.0
    if amin is None:
        return min(float(amax) * 0.5, 100_000.0)
    if amax is None:
        return max(float(amin) * 1.5, 50_000.0)
    return float((float(amin) + float(amax)) / 2)

def synthetic_names_from_keyword(keyword):
    keyword = normalize_text(keyword)
    if len(keyword) <= 2:
        return []
    base = keyword.title()
    names = [
        base,
        f'{base} Nusantara',
        f'{base} Sejahtera',
    ]
    if any(x in keyword for x in ['kopi', 'coffee', 'cafe', 'kafe']):
        names += [f'Kedai {base}', f'{base} Corner']
    if any(x in keyword for x in ['makan', 'resto', 'restaurant', 'restoran', 'nasi', 'bakso', 'sate']):
        names += [f'{base} Bu Ani', f'{base} Pak Budi']
    return list(dict.fromkeys(names))

def generate_synthetic_from_dictionary(max_per_category=250, max_keywords_per_rule=3):
    rows = []
    per_cat = Counter()

    for rule in DICTIONARY_RULES:
        detail = rule['category_detail']
        if detail not in VALID_CATEGORIES:
            continue
        if not rule['is_trusted_guardrail']:
            continue
        if per_cat[detail] >= max_per_category:
            continue

        amount = sample_amount_for_rule(rule)
        keywords = rule.get('alias_keywords', [])[:max_keywords_per_rule]
        for kw in keywords:
            for name in synthetic_names_from_keyword(kw):
                if per_cat[detail] >= max_per_category:
                    break
                rows.append({
                    'merchant': name,
                    'amount': amount,
                    'category_detail': detail,
                    'payment_method': 'qris',
                    'payment_media': 'mobile_banking',
                    'source': 'synthetic_dictionary',
                    'date': '2026-01-15 12:00:00',
                    'rolling_7d_spending': 0.0,
                    'rolling_30d_spending': 0.0,
                    'transaction_count_to_date': 1.0,
                    'sample_origin': 'synthetic_dictionary',
                    'base_sample_weight': 0.65,
                })
                per_cat[detail] += 1

    return prepare_features(pd.DataFrame(rows)) if rows else pd.DataFrame()

def typo_noise(text):
    text = str(text)
    if len(text) < 5:
        return text
    chars = list(text)
    op = random.choice(['delete', 'swap', 'drop_space'])
    if op == 'delete':
        idx = random.randrange(len(chars))
        del chars[idx]
        return ''.join(chars)
    if op == 'swap' and len(chars) > 4:
        idx = random.randrange(len(chars)-1)
        chars[idx], chars[idx+1] = chars[idx+1], chars[idx]
        return ''.join(chars)
    return text.replace(' ', '')

def generate_typo_augmentation(train_df, fraction=0.15, max_rows=1500):
    n = min(int(len(train_df) * fraction), max_rows)
    if n <= 0:
        return pd.DataFrame()
    sample = train_df.sample(n=n, random_state=SEED).copy()
    sample['merchant'] = sample['merchant'].apply(typo_noise)
    sample['sample_origin'] = 'typo_augmented'
    sample['base_sample_weight'] = 0.75
    return prepare_features(sample)

synthetic_df = generate_synthetic_from_dictionary(max_per_category=250)
typo_df = generate_typo_augmentation(train_df, fraction=0.15, max_rows=1500)

train_model_df = pd.concat([train_df, synthetic_df, typo_df], ignore_index=True)

print('Real train:', train_df.shape)
print('Synthetic:', synthetic_df.shape)
print('Typo aug :', typo_df.shape)
print('Train model:', train_model_df.shape)
print(train_model_df['sample_origin'].value_counts())

Real train: (6202, 51)
Synthetic: (2500, 21)
Typo aug : (930, 51)
Train model: (9632, 51)
sample_origin
real                    6202
synthetic_dictionary    2500
typo_augmented           930
Name: count, dtype: int64


## 10. Label encoding dan numeric normalization

In [10]:
LABEL_CLASSES = sorted(VALID_CATEGORIES)
label_to_id = {label: idx for idx, label in enumerate(LABEL_CLASSES)}
id_to_label = {idx: label for label, idx in label_to_id.items()}
NUM_CLASSES = len(LABEL_CLASSES)

for name, df in [('train_model_df', train_model_df), ('val_df', val_df), ('cold_test_df', cold_test_df)]:
    missing = sorted(set(df['category_detail']) - set(LABEL_CLASSES))
    if missing:
        raise ValueError(f'{name} punya label tidak dikenal: {missing}')

# Numeric stats fit hanya dari training data.
numeric_mean = train_model_df[NUMERIC_FEATURES].astype(float).mean().to_dict()
numeric_std = train_model_df[NUMERIC_FEATURES].astype(float).std().replace(0, 1).fillna(1).to_dict()

def add_normalized_numeric(df):
    df = df.copy()
    for col in NUMERIC_FEATURES:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)
        df[col + '_norm'] = (df[col] - numeric_mean[col]) / numeric_std[col]
    return df

train_model_df = add_normalized_numeric(train_model_df)
val_df = add_normalized_numeric(val_df)
cold_test_df = add_normalized_numeric(cold_test_df)

NORMALIZED_NUMERIC_FEATURES = [c + '_norm' for c in NUMERIC_FEATURES]

def encode_labels(df):
    return df['category_detail'].map(label_to_id).astype('int32').values

y_train = encode_labels(train_model_df)
y_val = encode_labels(val_df)
y_cold = encode_labels(cold_test_df)

# Class weight dikombinasikan dengan sample origin weight.
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train,
)
class_weight_map = {i: float(w) for i, w in enumerate(class_weights)}

train_sample_weight = np.array([
    float(train_model_df.iloc[i].get('base_sample_weight', 1.0)) * class_weight_map[int(y_train[i])]
    for i in range(len(train_model_df))
], dtype='float32')

print('Label classes:', LABEL_CLASSES)
print('Class weights:', class_weight_map)
print('Sample weight range:', float(train_sample_weight.min()), float(train_sample_weight.max()))

Label classes: ['education', 'entertainment', 'food', 'groceries', 'health', 'investment', 'self_care', 'shopping', 'transport', 'travel', 'utilities']
Class weights: {0: 0.9256198347107438, 1: 1.067849223946785, 2: 0.9838610827374872, 3: 1.0007272727272727, 4: 0.9217224880382775, 5: 0.9335142469470827, 6: 1.4741352923171105, 7: 0.9159376188664892, 8: 0.9445915465332941, 9: 1.0461605300314978, 10: 0.97401152796036}
Sample weight range: 0.5953594446182251 1.4741352796554565


## 11. Build input dictionary dan adapt vectorizer

In [11]:
def dataframe_to_inputs(df):
    inputs = {
        'merchant_text': df['merchant_clean'].astype(str).values,
        'numeric_input': df[NORMALIZED_NUMERIC_FEATURES].astype('float32').values,
    }
    for col in CATEGORICAL_FEATURES:
        inputs[f'{col}_input'] = df[col].astype(str).values
    return inputs

X_train = dataframe_to_inputs(train_model_df)
X_val = dataframe_to_inputs(val_df)
X_cold = dataframe_to_inputs(cold_test_df)

# Vectorizer akan masuk ke dalam model, sehingga preprocessing teks ikut tersimpan di .keras.
word_vectorizer = layers.TextVectorization(
    max_tokens=15_000,
    output_mode='int',
    output_sequence_length=12,
    standardize=None,
    name='word_vectorizer',
)
char_vectorizer = layers.TextVectorization(
    max_tokens=250,
    output_mode='int',
    output_sequence_length=48,
    split='character',
    standardize=None,
    name='char_vectorizer',
)

word_vectorizer.adapt(train_model_df['merchant_clean'].astype(str).values)
char_vectorizer.adapt(train_model_df['merchant_clean'].astype(str).values)

cat_lookups = {}
for col in CATEGORICAL_FEATURES:
    lookup = layers.StringLookup(mask_token=None, num_oov_indices=1, name=f'{col}_lookup')
    lookup.adapt(train_model_df[col].astype(str).values)
    cat_lookups[col] = lookup

print('Word vocab size:', len(word_vectorizer.get_vocabulary()))
print('Char vocab size:', len(char_vectorizer.get_vocabulary()))
for col, lookup in cat_lookups.items():
    print(col, len(lookup.get_vocabulary()))

Word vocab size: 1345
Char vocab size: 37
payment_method 6
payment_media 18
source 11
day_of_week 8
time_of_day 5
spending_level 4
is_weekend 3


## 12. Custom Layer, Custom Loss, Custom Callback

In [12]:
@tf.keras.utils.register_keras_serializable(package='SADAR')
class GatedFeatureFusion(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gate_dense = None

    def build(self, input_shape):
        total_dim = int(sum(shape[-1] for shape in input_shape))
        self.gate_dense = layers.Dense(total_dim, activation='sigmoid')
        super().build(input_shape)

    def call(self, inputs):
        x = tf.concat(inputs, axis=-1)
        gate = self.gate_dense(x)
        return x * gate

    def get_config(self):
        return super().get_config()

@tf.keras.utils.register_keras_serializable(package='SADAR')
class SparseCategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, from_logits=False, name='sparse_categorical_focal_loss', reduction='sum_over_batch_size', **kwargs):
        super().__init__(name=name, reduction=reduction, **kwargs)
        self.gamma = gamma
        self.from_logits = from_logits

    def call(self, y_true, y_pred):
        y_true = tf.cast(tf.reshape(y_true, [-1]), tf.int32)
        if self.from_logits:
            y_pred = tf.nn.softmax(y_pred, axis=-1)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=False)
        y_onehot = tf.one_hot(y_true, depth=tf.shape(y_pred)[-1])
        pt = tf.reduce_sum(y_onehot * y_pred, axis=-1)
        return tf.pow(1.0 - pt, self.gamma) * ce

    def get_config(self):
        config = super().get_config()
        config.update({'gamma': self.gamma, 'from_logits': self.from_logits})
        return config

class MacroF1Checkpoint(keras.callbacks.Callback):
    def __init__(self, validation_data, save_path):
        super().__init__()
        self.X_val, self.y_val = validation_data
        self.save_path = save_path
        self.best_f1 = -1.0

    def on_epoch_end(self, epoch, logs=None):
        proba = self.model.predict(self.X_val, verbose=0)
        pred = np.argmax(proba, axis=1)
        macro_f1 = f1_score(self.y_val, pred, average='macro')
        logs = logs or {}
        logs['val_macro_f1_external'] = macro_f1
        print(f' - val_macro_f1_external: {macro_f1:.4f}')
        if macro_f1 > self.best_f1:
            self.best_f1 = macro_f1
            self.model.save(self.save_path)
            print(f'   Saved best model to {self.save_path}')

## 13. Build Deep Learning model: word branch + char branch + numeric + categorical

In [13]:
def build_model():
    merchant_input = keras.Input(shape=(1,), dtype=tf.string, name='merchant_text')
    numeric_input = keras.Input(shape=(len(NORMALIZED_NUMERIC_FEATURES),), dtype=tf.float32, name='numeric_input')

    # Word-level merchant branch
    word_ids = word_vectorizer(merchant_input)
    word_emb = layers.Embedding(input_dim=len(word_vectorizer.get_vocabulary()), output_dim=64, name='word_embedding')(word_ids)
    word_x = layers.Bidirectional(layers.LSTM(48, return_sequences=True), name='word_bilstm')(word_emb)
    word_x = layers.GlobalMaxPooling1D(name='word_pool')(word_x)

    # Character-level merchant branch untuk merchant baru/typo.
    char_ids = char_vectorizer(merchant_input)
    char_emb = layers.Embedding(input_dim=len(char_vectorizer.get_vocabulary()), output_dim=32, name='char_embedding')(char_ids)
    char_x = layers.Conv1D(64, 3, activation='relu', padding='same', name='char_conv3')(char_emb)
    char_x = layers.Conv1D(64, 5, activation='relu', padding='same', name='char_conv5')(char_x)
    char_x = layers.GlobalMaxPooling1D(name='char_pool')(char_x)

    # Numeric branch
    num_x = layers.Dense(32, activation='relu', name='numeric_dense_1')(numeric_input)
    num_x = layers.BatchNormalization(name='numeric_bn')(num_x)
    num_x = layers.Dense(16, activation='relu', name='numeric_dense_2')(num_x)

    # Categorical branch
    cat_inputs = []
    cat_embeds = []
    for col in CATEGORICAL_FEATURES:
        inp = keras.Input(shape=(1,), dtype=tf.string, name=f'{col}_input')
        lookup = cat_lookups[col]
        ids = lookup(inp)
        vocab_size = len(lookup.get_vocabulary())
        dim = min(16, max(4, int(math.sqrt(vocab_size)) + 2))
        emb = layers.Embedding(input_dim=vocab_size, output_dim=dim, name=f'{col}_embedding')(ids)
        emb = layers.Flatten(name=f'{col}_flatten')(emb)
        cat_inputs.append(inp)
        cat_embeds.append(emb)

    cat_x = layers.Concatenate(name='categorical_concat')(cat_embeds) if len(cat_embeds) > 1 else cat_embeds[0]
    cat_x = layers.Dense(32, activation='relu', name='categorical_dense')(cat_x)

    fused = GatedFeatureFusion(name='gated_feature_fusion')([word_x, char_x, num_x, cat_x])
    x = layers.Dense(128, activation='relu', name='fusion_dense_1')(fused)
    x = layers.Dropout(0.30, name='dropout_1')(x)
    x = layers.Dense(64, activation='relu', name='fusion_dense_2')(x)
    x = layers.Dropout(0.20, name='dropout_2')(x)
    output = layers.Dense(NUM_CLASSES, activation='softmax', name='category_detail')(x)

    model = keras.Model(inputs=[merchant_input, numeric_input] + cat_inputs, outputs=output, name='SADAR_Merchant_Detail_Classifier')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss=SparseCategoricalFocalLoss(gamma=2.0),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')],
    )
    return model

model = build_model()
model.summary()

Model: "SADAR_Merchant_Detail_Classifier"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ merchant_text       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ payment_method_inp… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ payment_media_input │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_input        │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_of_week_input   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_of_day_input   │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spending_level_inp… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ is_weekend_input    │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ char_vectorizer     │ (None, 48)        │          0 │ merchant_text[0]… │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ payment_method_loo… │ (None, 1)         │          0 │ payment_method_i… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ payment_media_look… │ (None, 1)         │          0 │ payment_media_in… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ source_lookup       │ (None, 1)         │          0 │ source_input[0][… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ day_of_week_lookup  │ (None, 1)         │          0 │ day_of_week_inpu… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ time_of_day_lookup  │ (None, 1)         │          0 │ time_of_day_inpu… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spending_level_loo… │ (None, 1)         │          0 │ spending_level_i… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ is_weekend_lookup   │ (None, 1)         │          0 │ is_weekend_input… │
│ (StringLookup)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ word_vectorizer     │ (None, 12)        │          0 │ merchant_text[0]

 Total params: 238,838 (932.96 KB)

 Trainable params: 238,774 (932.71 KB)

 Non-trainable params: 64 (256.00 B)

## 14. Training

In [14]:
BEST_MODEL_PATH = str(MODEL_DIR / 'best_macro_f1_model.keras')

callbacks = [
    MacroF1Checkpoint(validation_data=(X_val, y_val), save_path=BEST_MODEL_PATH),
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
]

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    sample_weight=train_sample_weight,
    epochs=25,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

if Path(BEST_MODEL_PATH).exists():
    model = keras.models.load_model(
        BEST_MODEL_PATH,
        custom_objects={'GatedFeatureFusion': GatedFeatureFusion, 'SparseCategoricalFocalLoss': SparseCategoricalFocalLoss},
        compile=False,
    )
    print('Loaded best macro F1 model:', BEST_MODEL_PATH)

Epoch 1/25
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.1381 - loss: 1.7094 - val_macro_f1_external: 0.2332
   Saved best model to /content/sadar_merchant_classifier_production/model/best_macro_f1_model.keras
76/76 ━━━━━━━━━━━━━━━━━━━━ 19s 64ms/step - accuracy: 0.2214 - loss: 1.5180 - val_accuracy: 0.2766 - val_loss: 1.5145 - val_macro_f1_external: 0.2332 - learning_rate: 0.0010
Epoch 2/25
76/76 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 0.6285 - loss: 0.5309 - val_macro_f1_external: 0.6675
   Saved best model to /content/sadar_merchant_classifier_production/model/best_macro_f1_model.keras
76/76 ━━━━━━━━━━━━━━━━━━━━ 14s 73ms/step - accuracy: 0.7364 - loss: 0.3536 - val_accuracy: 0.6989 - val_loss: 0.8916 - val_macro_f1_external: 0.6675 - learning_rate: 0.0010
Epoch 3/25
74/76 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.9074 - loss: 0.1090 - val_macro_f1_external: 0.7229
   Saved best model to /content/sadar_merchant_classifier_production/model/best_macro_f1_model.keras


## 15. Evaluasi validation dan cold merchant test

In [15]:
def evaluate_model(df, X, y, name):
    proba = model.predict(X, verbose=0)
    pred = np.argmax(proba, axis=1)
    conf = np.max(proba, axis=1)

    print(f'===== {name} =====')
    print('Accuracy:', accuracy_score(y, pred))
    print('Macro F1:', f1_score(y, pred, average='macro'))
    print(classification_report(y, pred, target_names=LABEL_CLASSES, zero_division=0))

    result_df = df[['merchant', 'merchant_clean', 'category_detail', 'amount']].copy()
    result_df['pred_detail'] = [id_to_label[int(i)] for i in pred]
    result_df['confidence'] = conf
    result_df['correct'] = result_df['category_detail'] == result_df['pred_detail']
    return result_df, proba

val_result_df, val_proba = evaluate_model(val_df, X_val, y_val, 'Validation Unseen Merchants')
cold_result_df, cold_proba = evaluate_model(cold_test_df, X_cold, y_cold, 'Cold Merchant Test')

display(cold_result_df.sort_values('confidence').head(20))

===== Validation Unseen Merchants =====
Accuracy: 0.800702370500439
Macro F1: 0.7758515849605376
               precision    recall  f1-score   support

    education       0.86      1.00      0.92       148
entertainment       0.86      0.96      0.91        95
         food       0.69      0.89      0.77        97
    groceries       0.52      1.00      0.68       119
       health       1.00      1.00      1.00        62
   investment       0.91      1.00      0.95        96
    self_care       0.00      0.00      0.00       119
     shopping       0.94      0.66      0.78       100
    transport       0.94      0.51      0.66        94
       travel       0.85      0.94      0.89       125
    utilities       1.00      0.94      0.97        84

     accuracy                           0.80      1139
    macro avg       0.78      0.81      0.78      1139
 weighted avg       0.75      0.80      0.76      1139

===== Cold Merchant Test =====
Accuracy: 0.6769472033455306
Macro F1: 0.626

,merchant,merchant_clean,category_detail,amount,pred_detail,confidence,correct
705,Emina Cosmetics,emina cosmetics,self_care,432500.0,education,0.213739,False
1585,Emina Cosmetics,emina cosmetics,self_care,465000.0,self_care,0.215538,True
1420,Barbershop,barbershop,self_care,456500.0,groceries,0.217679,False
69,Barbershop,barbershop,self_care,500000.0,investment,0.232512,False
1422,Emina Cosmetics,emina cosmetics,self_care,369000.0,education,0.236775,False
1870,Udemy,udemy,education,376500.0,investment,0.236868,False
442,Barbershop,barbershop,self_care,342500.0,investment,0.239197,False
654,Emina Cosmetics,emina cosmetics,self_care,440000.0,self_care,0.239366,True
143,Emina Cosmetics,emina cosmetics,self_care,395000.0,education,0.240330,False
345,Emina Cosmetics,emina cosmetics,self_care,389500.0,self_care,0.242803,True


## 16. Threshold calibration: kenapa default 0.60

In [16]:
def threshold_report(result_df, thresholds=(0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80)):
    rows = []
    for th in thresholds:
        accepted = result_df['confidence'] >= th
        coverage = accepted.mean()
        acc_accepted = result_df.loc[accepted, 'correct'].mean() if accepted.any() else np.nan
        review_rate = 1.0 - coverage
        rows.append({
            'threshold': th,
            'coverage_auto_classified': coverage,
            'review_rate': review_rate,
            'accuracy_on_auto_classified': acc_accepted,
            'accepted_count': int(accepted.sum()),
            'review_count': int((~accepted).sum()),
        })
    return pd.DataFrame(rows)

threshold_df = threshold_report(cold_result_df)
display(threshold_df)
print('MODEL_REVIEW_THRESHOLD default:', MODEL_REVIEW_THRESHOLD)

,threshold,coverage_auto_classified,review_rate,accuracy_on_auto_classified,accepted_count,review_count
0,0.50,0.829587,0.170413,0.744171,1587,326
1,0.55,0.761108,0.238892,0.767170,1456,457
2,0.60,0.687402,0.312598,0.794677,1315,598
3,0.65,0.613173,0.386827,0.823529,1173,740
4,0.70,0.558808,0.441192,0.847521,1069,844
5,0.75,0.503398,0.496602,0.876428,963,950
6,0.80,0.448510,0.551490,0.903263,858,1055


MODEL_REVIEW_THRESHOLD default: 0.6


## 17. Export model dan artifact production

In [17]:
# Final model .keras
FINAL_MODEL_PATH = MODEL_DIR / MODEL_KERAS_FILE
model.save(FINAL_MODEL_PATH)
print('Saved .keras:', FINAL_MODEL_PATH)

# SavedModel untuk requirement siap produksi.
SAVEDMODEL_PATH = MODEL_DIR / MODEL_SAVEDMODEL_DIR
if SAVEDMODEL_PATH.exists():
    shutil.rmtree(SAVEDMODEL_PATH)
try:
    model.export(str(SAVEDMODEL_PATH))
except Exception:
    tf.saved_model.save(model, str(SAVEDMODEL_PATH))
print('SavedModel:', SAVEDMODEL_PATH)

# Dictionary source CSV.
# Production API memakai artifacts/dictionary_rules.json.
# CSV tetap ikut dicopy sebagai source of truth agar mudah diaudit/edit.
shutil.copy2(DICTIONARY_CSV_PATH, DICTIONARY_DIR / DICTIONARY_CSV_PATH.name)
shutil.copy2(DICTIONARY_CSV_PATH, DICTIONARY_DIR / 'merchant_dictionary_validator.csv')

# Artifact JSON
with open(ARTIFACTS_DIR / 'dictionary_rules.json', 'w', encoding='utf-8') as f:
    json.dump({
        'version': 'production_guardrail_v1',
        'trusted_rule_threshold': TRUSTED_RULE_THRESHOLD,
        'model_review_threshold': MODEL_REVIEW_THRESHOLD,
        'rules': DICTIONARY_RULES,
    }, f, ensure_ascii=False, indent=2)

with open(ARTIFACTS_DIR / 'category_primary_map.json', 'w', encoding='utf-8') as f:
    json.dump(CATEGORY_PRIMARY_MAP, f, ensure_ascii=False, indent=2)

with open(ARTIFACTS_DIR / 'label_classes.json', 'w', encoding='utf-8') as f:
    json.dump(LABEL_CLASSES, f, ensure_ascii=False, indent=2)

metadata = {
    'project': 'SADAR Merchant Classifier',
    'created_at': datetime.utcnow().isoformat() + 'Z',
    'model_keras_file': f'model/{MODEL_KERAS_FILE}',
    'saved_model_dir': f'model/{MODEL_SAVEDMODEL_DIR}',
    'dictionary_rules_file': 'artifacts/dictionary_rules.json',
    'label_classes_file': 'artifacts/label_classes.json',
    'category_primary_map_file': 'artifacts/category_primary_map.json',
    'category_detail_output': True,
    'category_primary_rule_based': True,
    'database_mapping': {'category_group': 'category_primary', 'category_detail': 'category_detail'},
    'trusted_rule_threshold': TRUSTED_RULE_THRESHOLD,
    'model_review_threshold': MODEL_REVIEW_THRESHOLD,
    'categorical_features': CATEGORICAL_FEATURES,
    'numeric_features': NUMERIC_FEATURES,
    'normalized_numeric_features': NORMALIZED_NUMERIC_FEATURES,
    'numeric_mean': numeric_mean,
    'numeric_std': numeric_std,
    'label_classes': LABEL_CLASSES,
    'decision_policy': [
        'If trusted dictionary rule matches with confidence >= 0.90, use rule.',
        'Otherwise use Deep Learning prediction.',
        'If model confidence < 0.60, return unknown_need_review.',
    ],
    'validation_metrics': {
        'val_accuracy': float(accuracy_score(y_val, np.argmax(val_proba, axis=1))),
        'val_macro_f1': float(f1_score(y_val, np.argmax(val_proba, axis=1), average='macro')),
        'cold_accuracy': float(accuracy_score(y_cold, np.argmax(cold_proba, axis=1))),
        'cold_macro_f1': float(f1_score(y_cold, np.argmax(cold_proba, axis=1), average='macro')),
    },
}

with open(ARTIFACTS_DIR / 'metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

# Sample transaksi dummy untuk testing API
sample_transactions = pd.DataFrame([
    {'merchant': 'Kopi Senja', 'amount': 35000, 'date': '2026-05-29 12:30:00', 'payment_method': 'QRIS', 'payment_media': 'mobile_banking', 'source': 'manual'},
    {'merchant': 'Toko Berkah', 'amount': 750000, 'date': '2026-05-29 21:00:00', 'payment_method': 'Debit', 'payment_media': 'BCA', 'source': 'manual'},
    {'merchant': 'Toko Obat Sehat', 'amount': 48000, 'date': '2026-05-29 10:00:00', 'payment_method': 'QRIS', 'payment_media': 'mobile_banking', 'source': 'manual'},
])
sample_transactions.to_csv(DATA_SAMPLE_DIR / 'sample_transactions.csv', index=False)

print('Artifacts exported to:', OUTPUT_DIR)
print(sorted([str(p.relative_to(OUTPUT_DIR)) for p in OUTPUT_DIR.rglob('*') if p.is_file()])[:30])

Saved .keras: /content/sadar_merchant_classifier_production/model/sadar_merchant_detail_classifier.keras
Saved artifact at '/content/sadar_merchant_classifier_production/model/sadar_merchant_detail_classifier_savedmodel'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 1), dtype=tf.string, name='merchant_text'), TensorSpec(shape=(None, 8), dtype=tf.float32, name='numeric_input'), TensorSpec(shape=(None, 1), dtype=tf.string, name='payment_method_input'), TensorSpec(shape=(None, 1), dtype=tf.string, name='payment_media_input'), TensorSpec(shape=(None, 1), dtype=tf.string, name='source_input'), TensorSpec(shape=(None, 1), dtype=tf.string, name='day_of_week_input'), TensorSpec(shape=(None, 1), dtype=tf.string, name='time_of_day_input'), TensorSpec(shape=(None, 1), dtype=tf.string, name='spending_level_input'), TensorSpec(shape=(None, 1), dtype=tf.string, name='is_weekend_input')]
Output Type:
  TensorSpec(shape=(None, 11),

/tmp/ipykernel_5876/2353604466.py:39: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat() + 'Z',


## 18. Generate `inference.py` production-ready

In [18]:
inference_py = r"""
from pathlib import Path
import re
import json
import math
import unicodedata
from typing import Dict, Any, Optional

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers


@tf.keras.utils.register_keras_serializable(package='SADAR')
class GatedFeatureFusion(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.gate_dense = None

    def build(self, input_shape):
        total_dim = int(sum(shape[-1] for shape in input_shape))
        self.gate_dense = layers.Dense(total_dim, activation='sigmoid')
        super().build(input_shape)

    def call(self, inputs):
        x = tf.concat(inputs, axis=-1)
        gate = self.gate_dense(x)
        return x * gate

    def get_config(self):
        return super().get_config()


def normalize_text(text):
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return 'unknown'
    text = str(text).lower().strip()
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(ch for ch in text if not unicodedata.combining(ch))
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else 'unknown'


def infer_spending_level(amount):
    amount = float(amount or 0)
    if amount < 50000:
        return 'low'
    if amount < 300000:
        return 'medium'
    return 'high'


def infer_time_of_day(hour):
    try:
        hour = int(hour)
    except Exception:
        hour = 0
    if 5 <= hour < 11:
        return 'pagi'
    if 11 <= hour < 15:
        return 'siang'
    if 15 <= hour < 18:
        return 'sore'
    return 'malam'


def keyword_matches(merchant_clean, keyword, match_type):
    if not keyword:
        return False
    keyword = normalize_text(keyword)
    if match_type == 'exact':
        return merchant_clean == keyword
    if match_type == 'starts_with':
        return merchant_clean.startswith(keyword)
    if len(keyword) <= 3:
        return re.search(rf'(?<![a-z0-9]){re.escape(keyword)}(?![a-z0-9])', merchant_clean) is not None
    return keyword in merchant_clean


class SADARMerchantClassifier:
    def __init__(self, base_dir: Optional[str] = None):
        self.base_dir = Path(base_dir) if base_dir else Path(__file__).resolve().parent
        self.metadata_path = self.base_dir / 'artifacts' / 'metadata.json'
        self.rules_path = self.base_dir / 'artifacts' / 'dictionary_rules.json'
        self.label_path = self.base_dir / 'artifacts' / 'label_classes.json'
        self.primary_map_path = self.base_dir / 'artifacts' / 'category_primary_map.json'

        self._load_artifacts()

    def _load_json(self, path):
        if not path.exists():
            raise FileNotFoundError(f'Required artifact not found: {path}')
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)

    def _load_artifacts(self):
        self.metadata = self._load_json(self.metadata_path)
        self.rules_payload = self._load_json(self.rules_path)
        self.dictionary_rules = self.rules_payload.get('rules', [])
        self.label_classes = self._load_json(self.label_path)
        self.category_primary_map = self._load_json(self.primary_map_path)

        self.trusted_rule_threshold = float(self.metadata.get('trusted_rule_threshold', 0.90))
        self.model_review_threshold = float(self.metadata.get('model_review_threshold', 0.60))
        self.numeric_features = self.metadata['numeric_features']
        self.categorical_features = self.metadata['categorical_features']
        self.numeric_mean = self.metadata['numeric_mean']
        self.numeric_std = self.metadata['numeric_std']

        model_path = self.base_dir / self.metadata['model_keras_file']
        if not model_path.exists():
            raise FileNotFoundError(f'Model .keras not found: {model_path}')
        self.model = keras.models.load_model(
            model_path,
            custom_objects={'GatedFeatureFusion': GatedFeatureFusion},
            compile=False,
        )

    def _amount_allowed(self, rule, amount):
        amount = float(amount or 0)
        amin = rule.get('amount_min')
        amax = rule.get('amount_max')
        if amin is not None and amount < float(amin):
            return False
        if amax is not None and amount > float(amax):
            return False
        return True

    def apply_dictionary_rules(self, merchant, amount=0):
        merchant_clean = normalize_text(merchant)
        for rule in self.dictionary_rules:
            if not rule.get('active', True):
                continue
            if any(keyword_matches(merchant_clean, neg, 'contains') for neg in rule.get('negative_keywords', [])):
                continue
            if not self._amount_allowed(rule, amount):
                continue
            match_type = rule.get('match_type', 'contains')
            matched_keyword = None
            for kw in rule.get('alias_keywords', []):
                if keyword_matches(merchant_clean, kw, match_type):
                    matched_keyword = kw
                    break
            if matched_keyword:
                return {
                    'rule_category': rule['category_detail'],
                    'rule_primary': rule.get('category_primary', self.category_primary_map.get(rule['category_detail'], 'unknown')),
                    'rule_confidence': float(rule.get('rule_confidence', 0.0)),
                    'matched_rule': rule.get('rule_id'),
                    'matched_keyword': matched_keyword,
                    'is_trusted_guardrail': bool(rule.get('is_trusted_guardrail', False)),
                    'rule_ambiguity_level': rule.get('ambiguity_level'),
                    'rule_strength': rule.get('rule_strength'),
                }
        return {
            'rule_category': None,
            'rule_primary': None,
            'rule_confidence': 0.0,
            'matched_rule': None,
            'matched_keyword': None,
            'is_trusted_guardrail': False,
            'rule_ambiguity_level': None,
            'rule_strength': None,
        }

    def _prepare_one(self, merchant, amount, date=None, payment_method='unknown', payment_media='unknown', source='manual'):
        amount = float(amount or 0)
        merchant_clean = normalize_text(merchant)
        dt = pd.to_datetime(date, errors='coerce') if date else pd.NaT
        if pd.isna(dt):
            month = 0
            day = 0
            week = 0
            day_of_week = 'unknown'
            is_weekend = 'false'
            time_of_day = 'unknown'
        else:
            month = int(dt.month)
            day = int(dt.day)
            week = int(dt.isocalendar().week)
            day_of_week = str(dt.day_name()).lower()
            is_weekend = str(dt.dayofweek >= 5).lower()
            time_of_day = infer_time_of_day(dt.hour)

        row = {
            'merchant_clean': merchant_clean,
            'amount': amount,
            'log_amount': float(np.log1p(max(amount, 0))),
            'month': month,
            'day': day,
            'week': week,
            'rolling_7d_spending': 0.0,
            'rolling_30d_spending': 0.0,
            'transaction_count_to_date': 1.0,
            'payment_method': normalize_text(payment_method),
            'payment_media': normalize_text(payment_media),
            'source': normalize_text(source),
            'day_of_week': day_of_week,
            'time_of_day': time_of_day,
            'spending_level': infer_spending_level(amount),
            'is_weekend': is_weekend,
        }
        return row

    def _model_inputs_from_row(self, row):
        numeric_values = []
        for col in self.numeric_features:
            value = float(row.get(col, 0.0))
            mean = float(self.numeric_mean.get(col, 0.0))
            std = float(self.numeric_std.get(col, 1.0)) or 1.0
            numeric_values.append((value - mean) / std)

        inputs = {
            'merchant_text': np.array([row['merchant_clean']], dtype=object),
            'numeric_input': np.array([numeric_values], dtype='float32'),
        }
        for col in self.categorical_features:
            inputs[f'{col}_input'] = np.array([str(row.get(col, 'unknown'))], dtype=object)
        return inputs

    def predict_one(self, merchant, amount, date=None, payment_method='unknown', payment_media='unknown', source='manual') -> Dict[str, Any]:
        row = self._prepare_one(merchant, amount, date, payment_method, payment_media, source)
        rule = self.apply_dictionary_rules(merchant, amount)

        inputs = self._model_inputs_from_row(row)
        proba = self.model.predict(inputs, verbose=0)[0]
        order = np.argsort(proba)[::-1]
        top_idx = int(order[0])
        second_idx = int(order[1]) if len(order) > 1 else top_idx
        ai_detail = self.label_classes[top_idx]
        ai_confidence = float(proba[top_idx])
        confidence_margin = float(proba[top_idx] - proba[second_idx])

        use_rule = (
            rule['is_trusted_guardrail']
            and rule['rule_confidence'] >= self.trusted_rule_threshold
            and rule['rule_category'] is not None
        )

        if use_rule:
            final_detail = rule['rule_category']
            final_primary = self.category_primary_map.get(final_detail, rule.get('rule_primary') or 'unknown')
            decision_source = 'trusted_dictionary_guardrail'
            confidence = float(rule['rule_confidence'])
            needs_review = False
        else:
            if ai_confidence < self.model_review_threshold:
                final_detail = 'unknown_need_review'
                final_primary = 'unknown'
                decision_source = 'model_low_confidence_review'
                confidence = ai_confidence
                needs_review = True
            else:
                final_detail = ai_detail
                final_primary = self.category_primary_map.get(final_detail, 'unknown')
                decision_source = 'deep_learning_model'
                confidence = ai_confidence
                needs_review = False

        return {
            'merchant': merchant,
            # Untuk backend/database SADAR:
            # category_group = primary category (Needs/Wants/Investment)
            # category_detail = detail category hasil AI/rule
            'category_group': final_primary,
            'category_detail': final_detail,
            'category_primary': final_primary,
            'confidence': round(float(confidence), 6),
            'needs_review': bool(needs_review),
            'decision_source': decision_source,
            'ai_detail': ai_detail,
            'ai_confidence': round(ai_confidence, 6),
            'confidence_margin': round(confidence_margin, 6),
            'rule_category': rule['rule_category'],
            'rule_confidence': round(float(rule['rule_confidence']), 6),
            'matched_rule': rule['matched_rule'],
            'matched_keyword': rule['matched_keyword'],
            'threshold_used': self.model_review_threshold,
            'trusted_rule_threshold_used': self.trusted_rule_threshold,
        }
"""

with open(OUTPUT_DIR / 'inference.py', 'w', encoding='utf-8') as f:
    f.write(inference_py)

print('Generated:', OUTPUT_DIR / 'inference.py')

Generated: /content/sadar_merchant_classifier_production/inference.py


## 19. Generate `main.py` FastAPI

In [19]:
main_py = r"""
from typing import Optional
from fastapi import FastAPI
from pydantic import BaseModel, Field
from inference import SADARMerchantClassifier

app = FastAPI(title='SADAR Merchant Classifier API', version='1.0.0')
classifier = SADARMerchantClassifier(base_dir='.')

class TransactionRequest(BaseModel):
    merchant: str = Field(..., example='Kopi Senja')
    amount: float = Field(..., example=35000)
    date: Optional[str] = Field(None, example='2026-05-29 12:30:00')
    payment_method: str = Field('unknown', example='QRIS')
    payment_media: str = Field('unknown', example='mobile_banking')
    source: str = Field('manual', example='manual')

@app.get('/')
def root():
    return {'status': 'ok', 'service': 'SADAR Merchant Classifier API'}

@app.get('/health')
def health():
    return {'status': 'healthy'}

@app.post('/predict')
def predict(payload: TransactionRequest):
    return classifier.predict_one(
        merchant=payload.merchant,
        amount=payload.amount,
        date=payload.date,
        payment_method=payload.payment_method,
        payment_media=payload.payment_media,
        source=payload.source,
    )
"""

with open(OUTPUT_DIR / 'main.py', 'w', encoding='utf-8') as f:
    f.write(main_py)

print('Generated:', OUTPUT_DIR / 'main.py')

Generated: /content/sadar_merchant_classifier_production/main.py


## 20. Generate requirements.txt, README, dan ZIP deployment

In [20]:
requirements_txt = """fastapi==0.115.0
uvicorn[standard]==0.30.6
tensorflow-cpu==2.17.0
numpy==1.26.4
pandas==2.2.2
scikit-learn==1.5.2
pydantic==2.8.2
"""

readme_md = f"""# SADAR Merchant Classifier API

Model Deep Learning untuk klasifikasi `category_detail` transaksi merchant. `category_primary` ditentukan secara rule-based melalui `category_primary_map.json` agar konsisten dengan aturan bisnis.

Untuk database SADAR:
- `category_group` = primary category (`Needs`, `Wants`, `Investment`)
- `category_detail` = detail category (`food`, `groceries`, `transport`, dst.)

Response API tetap menyertakan `category_primary` dan juga alias `category_group` agar mudah diintegrasikan dengan tabel `transactions`.

## Decision Policy

1. Jika rule cocok, aman, tidak ambigu, dan confidence >= {TRUSTED_RULE_THRESHOLD}, gunakan dictionary guardrail.
2. Jika rule tidak cukup kuat, gunakan prediksi Deep Learning.
3. Jika confidence model < {MODEL_REVIEW_THRESHOLD}, output menjadi `unknown_need_review`.

## Run Local

```bash
pip install -r requirements.txt
uvicorn main:app --host 0.0.0.0 --port 8000
```

Buka:

```text
http://localhost:8000/docs
```

## Render/Railway

Build command:

```bash
pip install -r requirements.txt
```

Start command:

```bash
uvicorn main:app --host 0.0.0.0 --port $PORT
```

## Example Request

```json
{{
  "merchant": "Kopi Senja",
  "amount": 35000,
  "date": "2026-05-29 12:30:00",
  "payment_method": "QRIS",
  "payment_media": "mobile_banking",
  "source": "manual"
}}
```
"""

with open(OUTPUT_DIR / 'requirements.txt', 'w', encoding='utf-8') as f:
    f.write(requirements_txt)

with open(OUTPUT_DIR / 'README.md', 'w', encoding='utf-8') as f:
    f.write(readme_md)

# Render/Railway friendly Python version marker.
with open(OUTPUT_DIR / '.python-version', 'w', encoding='utf-8') as f:
    f.write('3.11.9\n')

# Git ignore untuk folder deployment.
gitignore_txt = """__pycache__/
*.pyc
.ipynb_checkpoints/
*.log
*.zip
.env
venv/
.env.local

# Jangan commit full raw training dataset ke folder deployment
data_budget_modelling.csv
budget_management.csv
budget_management_clean.csv
"""
with open(OUTPUT_DIR / '.gitignore', 'w', encoding='utf-8') as f:
    f.write(gitignore_txt)

# Zip output folder untuk download mudah dari Colab.
zip_base = Path('/content/sadar_merchant_classifier_production_artifacts')
if zip_base.with_suffix('.zip').exists():
    zip_base.with_suffix('.zip').unlink()
shutil.make_archive(str(zip_base), 'zip', root_dir=OUTPUT_DIR)

print('Generated requirements, README, ZIP')
print('Folder:', OUTPUT_DIR)
print('ZIP   :', zip_base.with_suffix('.zip'))

Generated requirements, README, ZIP
Folder: /content/sadar_merchant_classifier_production
ZIP   : /content/sadar_merchant_classifier_production_artifacts.zip


## 21. Demo inference merchant baru langsung di notebook

In [21]:
import sys
sys.path.insert(0, str(OUTPUT_DIR))

from inference import SADARMerchantClassifier

classifier = SADARMerchantClassifier(base_dir=str(OUTPUT_DIR))

new_merchant_tests = [
    {'merchant': 'Soto Segar Ibu', 'amount': 45000, 'date': '2026-05-29 12:30:00', 'payment_method': 'QRIS', 'payment_media': 'mobile_banking', 'source': 'manual'},
    {'merchant': 'Apotek sehat bu ani', 'amount': 68000, 'date': '2026-05-29 10:20:00', 'payment_method': 'QRIS', 'payment_media': 'mobile_banking', 'source': 'manual'},
    {'merchant': 'Warung Makan Bu Ani', 'amount': 27000, 'date': '2026-05-29 12:10:00', 'payment_method': 'Cash', 'payment_media': 'cash', 'source': 'manual'},
    {'merchant': 'Toko Obat Sehat', 'amount': 48000, 'date': '2026-05-29 10:00:00', 'payment_method': 'QRIS', 'payment_media': 'mobile_banking', 'source': 'manual'},
    {'merchant': 'Toko Buku Cerdas', 'amount': 95000, 'date': '2026-05-29 15:00:00', 'payment_method': 'Debit', 'payment_media': 'BCA', 'source': 'manual'},
    {'merchant': 'Koperasi Merah Putih', 'amount': 65000, 'date': '2026-05-29 13:00:00', 'payment_method': 'QRIS', 'payment_media': 'mobile_banking', 'source': 'manual'},
    {'merchant': 'Toko Sembako', 'amount': 780000, 'date': '2026-05-29 10:00:00', 'payment_method': 'Debit', 'payment_media': 'BCA', 'source': 'manual'},
    {'merchant': 'PT Maju Sejahtera Abadi', 'amount': 1250000, 'date': '2026-05-29 09:00:00', 'payment_method': 'Transfer', 'payment_media': 'bank_transfer', 'source': 'manual'},
]

results = [classifier.predict_one(**x) for x in new_merchant_tests]
display(pd.DataFrame(results)[['merchant','category_group','category_detail','confidence','needs_review','decision_source','ai_detail','rule_category','matched_keyword']])

,merchant,category_group,category_detail,confidence,needs_review,decision_source,ai_detail,rule_category,matched_keyword
0,Soto Segar Ibu,Needs,food,0.990000,False,trusted_dictionary_guardrail,groceries,food,soto
1,Apotek sehat bu ani,Needs,health,0.990000,False,trusted_dictionary_guardrail,entertainment,health,apotek
2,Warung Makan Bu Ani,Needs,food,0.990000,False,trusted_dictionary_guardrail,food,food,warung makan
3,Toko Obat Sehat,Needs,health,0.990000,False,trusted_dictionary_guardrail,food,health,toko obat
4,Toko Buku Cerdas,Wants,education,0.990000,False,trusted_dictionary_guardrail,education,education,toko buku
5,Koperasi Merah Putih,Needs,food,0.907977,False,deep_learning_model,food,investment,koperasi
6,Toko Sembako,Needs,groceries,0.990000,False,trusted_dictionary_guardrail,groceries,groceries,toko sembako
7,PT Maju Sejahtera Abadi,unknown,unknown_need_review,0.303019,True,model_low_confidence_review,education,None,None


## 22. File yang perlu masuk GitHub

In [22]:
print('Upload folder ini ke GitHub / Render Root Directory:')
print(OUTPUT_DIR)
print('\nStruktur penting:')
for p in sorted(OUTPUT_DIR.rglob('*')):
    if p.is_file():
        rel = p.relative_to(OUTPUT_DIR)
        if not str(rel).startswith('model/sadar_merchant_detail_classifier_savedmodel'):
            print('-', rel)

print('\nRender Root Directory kalau folder ini berada di repo utama: ai/merchant-classifier')
print('Build Command : pip install -r requirements.txt')
print('Start Command : uvicorn main:app --host 0.0.0.0 --port $PORT')

Upload folder ini ke GitHub / Render Root Directory:
/content/sadar_merchant_classifier_production

Struktur penting:
- .gitignore
- .python-version
- README.md
- __pycache__/inference.cpython-312.pyc
- artifacts/category_primary_map.json
- artifacts/dictionary_rules.json
- artifacts/label_classes.json
- artifacts/metadata.json
- data/sample_transactions.csv
- dictionary/merchant_dictionary_validator.csv
- inference.py
- main.py
- model/best_macro_f1_model.keras
- model/sadar_merchant_detail_classifier.keras
- requirements.txt

Render Root Directory kalau folder ini berada di repo utama: ai/merchant-classifier
Build Command : pip install -r requirements.txt
Start Command : uvicorn main:app --host 0.0.0.0 --port $PORT
